## This notebook is to derive channel masks from rapid-repeat SAR (UAVSAR) coherence
- as defined in section 4.2 of Varugu et al. (2025) DeltaX UAVSAR channel masks paper.

In [ ]:
import os
import glob
import numpy as np
from osgeo import gdal
import matplotlib.pyplot as plt
import warnings
from mintpy.utils import ptime, readfile, writefile, utils as ut
from mintpy.objects import ifgramStack
import imageio
import matplotlib as mpl

## Input file paths
- Download the data from zenodo (https://zenodo.org/records/14047957). 
- Unzip the folder and update the path to data_folder


In [ ]:
data_folder = '/path/to/Atchafalaya_UAVSAR_interferograms_coherence';
coherence_file_list = sorted(glob.glob(os.path.join(data_folder,'gulfco_12011_161017*_geo_coh.tif')));

## Read the first file for sizegeo info 

In [ ]:
ds = gdal.Open(coherence_file_list[0])
if ds:
    coh = ds.ReadAsArray();
[lines, samples] = np.shape(coh);
print('Lines:',lines, 'Samples:',samples);
# Extract geotransform and projection from the coherence file
geotransform = ds.GetGeoTransform();
projection = ds.GetProjection();

## Read coherence 
- Load data from all file in one 3D-array
- Make list of indices for NN (30 min) and NN12 (60,90min) files;

In [ ]:
dateList = []; NN= []; NN12 = [];
coherence = np.zeros((len(coherence_file_list),lines, samples), dtype=np.float32) * np.nan
for i,file in enumerate(coherence_file_list):
    ds = gdal.Open(coherence_file_list[i])
    if ds:
        coh = ds.ReadAsArray();
        coherence[i,:,:] = coh
    date = '20'+os.path.basename(file)[13:24]+'20'+os.path.basename(file)[24:34];
    dateList.append(date);
    date1, date2 = date.split('_'); 
    date1 = dt.datetime.strptime(date1, "%Y%m%d%H%M");
    date2 = dt.datetime.strptime(date2, "%Y%m%d%H%M");
    diff = date2-date1; diff = (diff.seconds)/60;
    if diff<=30:
        NN.append(i);
    elif diff<=90:
        NN12.append(i);
    else:
        pass

## Calculate mean coherence

In [ ]:
mean_coh= np.mean(coherence[NN],axis=0);
mean_coh12= np.mean(coherence[NN12],axis=0);

## Plot coherence histograms
- Select the optimal threshold from the coherence histograms.
- Options include to compute coherence using Otsu's method.

In [ ]:
%matplotlib inline
fig,axes = plt.subplots(1,2,figsize=(10,5),sharey=True);
ax1=axes[0];ax2=axes[1];
ax1.tick_params(labelsize=15);ax2.tick_params(labelsize=15);
ax2.tick_params(axis='x',length=15, width=5); 
ax1.tick_params(axis='x',length=15, width=5); 

ax1.tick_params(axis='y',length=15, width=5);
ax2.tick_params(axis='y',length=0, width=0);


cax1=ax1.hist(mean_coh.flatten(),bins=np.linspace(0,1,500));

cax2=ax2.hist(mean_coh12.flatten(),bins=np.linspace(0,1,500));
# if using Otsu's threshold
# NNcoh_threshold = skimage.filters.threshold_otsu(mean_coh);
NNcoh_threshold = 0.4;
print("threshold for mean_coh = {}.".format(NNcoh_threshold))
ax1.axvline(NNcoh_threshold,color='red')
# if using Otsu's threshold
#t = skimage.filters.threshold_otsu(mean_coh12);
NN12coh_threshold = 0.5;
print("threshold for mean_coh12 = {}.".format(NN12coh_threshold))
ax2.axvline(NN12coh_threshold,color='red')

ax1.set_ylim([0,1e6]);
#plt.savefig('mean_cohNN_mean_coh_NN12_histograms.pdf',bbox_inches='tight',transparent=True);

## Generate mask
- Generate 3 category mask. 
- 0 - Open Water; 1 - Wetland; 2 - Intermittent flow

In [ ]:
myMask1=np.ones([lines, samples]);
#map open water
myMask1[np.where( (mean_coh<NNcoh_threshold) ) ]=0;
#map intermittent flow
myMask1[np.where((mean_coh > NNcoh_threshold) & (mean_coh12 < NN12coh_threshold))] = 2;
myMask1[np.where(np.isnan(coh))] = np.nan;
plt.imshow(myMask1, cmap='jet',vmin=0,vmax=2);

In [ ]:
# Create a empty dataset for the mask with the same georeferencing as the coherence file
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create('high_tide_channel_mask.tif', samples, lines, 1, gdal.GDT_Float32)
out_ds.SetGeoTransform(geotransform)
out_ds.SetProjection(projection)

# Write the geocoded mask data
out_ds.GetRasterBand(1).WriteArray(myMask1)

# Close the datasets
out_ds = None
ds = None

print("Done geocoding")